# Local Serbia RAG Smoke Test

This notebook runs the Serbia smoke workflow locally (no Cloud Run), writes local artifacts, and lets you run custom retrieval checks over the generated chunk set.

In [17]:
from __future__ import annotations

import json
import subprocess
import sys
from collections import defaultdict
from pathlib import Path

from src.embeddings.client import DeterministicEmbeddingClient
from src.retrieval.context_windows import RetrievalContextWindowExpander
from src.retrieval.hybrid import HybridRetriever
from src.retrieval.lexical import LexicalRetriever
from src.retrieval.semantic import SemanticRetriever
from src.retrieval.service import RetrievalService
from src.schemas.source_metadata import SourceChunk, SourceMetadata
from src.storage.sources import InMemorySourceRepository

In [18]:
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / ".local" / "serbia-smoke"
SCRIPT = ROOT / "scripts" / "local_serbia_rag_smoke.py"

print("ROOT:", ROOT)
print("Python:", sys.executable)
print("Data dir exists:", DATA_DIR.exists())
print("Smoke script exists:", SCRIPT.exists())

ROOT: /Users/sonle/Documents/work/WB/wb-ldt-app
Python: /Users/sonle/opt/anaconda3/envs/wb-ldt-app-backend/bin/python
Data dir exists: True
Smoke script exists: True


In [19]:
cmd = [
    sys.executable,
    str(SCRIPT),
    "--data-dir",
    str(DATA_DIR),
    "--output-dir",
    str(OUTPUT_DIR),
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

Running: /Users/sonle/opt/anaconda3/envs/wb-ldt-app-backend/bin/python /Users/sonle/Documents/work/WB/wb-ldt-app/scripts/local_serbia_rag_smoke.py --data-dir /Users/sonle/Documents/work/WB/wb-ldt-app/data --output-dir /Users/sonle/Documents/work/WB/wb-ldt-app/.local/serbia-smoke
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
{
  "load_summary": {
    "total_rows": 408,
    "family_counts": {
      "serbia_national_documents": 24,
      "serbia_municipal_development_plans": 161,
      "serbia_lsg_projects": 107,
      "serbia_wbif_projects": 77,
      "serbia_wbif_tas": 39
    }
  },
  "ingest_summary_counts": {
    "scanned_rows": 408,
    "ingested_document_rows": 2,
    "ingested_structured_rows": 406,
    "failed_rows": 0,
    "skipped_rows": 0
  },
  "chunk_count": 408,
  "municipal_result_count": 1,
  "national_result_count": 5,
  "municipal_top_source_ids": [
    "serbia-serbia_municipal_development_plans-municipal_development_plan-lu-ani-l

CompletedProcess(args=['/Users/sonle/opt/anaconda3/envs/wb-ldt-app-backend/bin/python', '/Users/sonle/Documents/work/WB/wb-ldt-app/scripts/local_serbia_rag_smoke.py', '--data-dir', '/Users/sonle/Documents/work/WB/wb-ldt-app/data', '--output-dir', '/Users/sonle/Documents/work/WB/wb-ldt-app/.local/serbia-smoke'], returncode=0)

In [20]:
smoke_report_path = OUTPUT_DIR / "smoke_report.json"
full_report_path = OUTPUT_DIR / "smoke_report_full.json"
chunks_path = OUTPUT_DIR / "chunks.json"

with smoke_report_path.open("r", encoding="utf-8") as f:
    smoke_report = json.load(f)

print(json.dumps(smoke_report, indent=2, ensure_ascii=False))
print()
print("Artifacts:")
print("-", smoke_report_path)
print("-", full_report_path)
print("-", chunks_path)

{
  "load_summary": {
    "total_rows": 408,
    "family_counts": {
      "serbia_national_documents": 24,
      "serbia_municipal_development_plans": 161,
      "serbia_lsg_projects": 107,
      "serbia_wbif_projects": 77,
      "serbia_wbif_tas": 39
    }
  },
  "ingest_summary_counts": {
    "scanned_rows": 408,
    "ingested_document_rows": 2,
    "ingested_structured_rows": 406,
    "failed_rows": 0,
    "skipped_rows": 0
  },
  "chunk_count": 408,
  "municipal_result_count": 1,
  "national_result_count": 5,
  "municipal_top_source_ids": [
    "serbia-serbia_municipal_development_plans-municipal_development_plan-lu-ani-local-development-plan-856fbb8551"
  ],
  "national_top_source_ids": [
    "serbia-serbia_national_documents-national_policy_document-urban-sustainable-and-integrated-development-031e25ffb8",
    "serbia-serbia_national_documents-national_policy_document-national-health-strategy-3d2cd6caeb",
    "serbia-serbia_national_documents-national_policy_document-integrated-n

In [12]:
with chunks_path.open("r", encoding="utf-8") as f:
    raw_chunks = json.load(f)

source_repo = InMemorySourceRepository()
grouped: dict[str, list[SourceChunk]] = defaultdict(list)

for payload in raw_chunks:
    chunk = SourceChunk.model_validate(payload)
    grouped[chunk.source_id].append(chunk)

for source_id, rows in grouped.items():
    rows = sorted(rows, key=lambda c: c.chunk_index)
    first = rows[0]
    source_repo.upsert_source(
        SourceMetadata(
            source_id=source_id,
            source_type=first.source_type,
            title=source_id,
            uri=f"local://{source_id}",
            municipality_id=first.municipality_id,
            category=first.category,
            mime_type="text/plain",
        )
    )
    source_repo.replace_chunks(source_id, rows)

embedding_client = DeterministicEmbeddingClient()
retrieval = RetrievalService(
    semantic_retriever=SemanticRetriever(source_repo, embedding_client),
    lexical_retriever=LexicalRetriever(source_repo),
    hybrid_retriever=HybridRetriever(
        lexical_retriever=LexicalRetriever(source_repo),
        semantic_retriever=SemanticRetriever(source_repo, embedding_client),
    ),
    context_window_expander=RetrievalContextWindowExpander(source_repo, neighbor_window=0),
)

print(f"Loaded {len(raw_chunks)} chunks across {len(grouped)} sources.")

Loaded 408 chunks across 408 sources.


In [13]:
def run_query(
    query: str,
    *,
    mode: str = "semantic",
    top_k: int = 5,
    source_types: set[str] | None = None,
    municipality_id: str | None = None,
    category: str | None = None,
):
    response = retrieval.search(
        query=query,
        mode=mode,
        top_k=top_k,
        source_types=source_types,
        municipality_id=municipality_id,
        category=category,
    )
    print(f"mode={mode} total_results={response.total_results}")
    print("query:", query)
    print("municipality_id:", municipality_id)
    print("category:", category)
    for idx, item in enumerate(response.results, start=1):
        snippet = " ".join((item.snippet or "").splitlines())[:220]
        title = item.citation_title or item.metadata.get("title") or "(no-title)"
        print(f"  {idx}. score={item.score:.4f} title={title}")
        print(f"     source_id={item.source_id} | chunk_id={item.chunk_id}")
        print(f"     snippet={snippet}")
    return response

In [14]:
available_municipality_ids = sorted({payload.get("municipality_id") for payload in raw_chunks if payload.get("municipality_id")})
print("Available municipality IDs (sample):", available_municipality_ids[:20])

Available municipality IDs (sample): ['srb-ada', 'srb-aleksandrovac', 'srb-aleksinac', 'srb-alibunar', 'srb-apatin', 'srb-aranelovac', 'srb-arilje', 'srb-babusnica', 'srb-bac', 'srb-backa-palanka', 'srb-backa-topola', 'srb-backi-petrovac', 'srb-bajina-basta', 'srb-batocina', 'srb-becej', 'srb-bela-crkva', 'srb-bela-palanka', 'srb-beocin', 'srb-beograd-barajevo', 'srb-beograd-cukarica']


In [16]:


# Set this to any value from available_municipality_ids, e.g. "srb-belgrade".
SELECTED_MUNICIPALITY_ID = 'srb-beocin'
print("Selected municipality:", SELECTED_MUNICIPALITY_ID)

municipal_query = "What are the key local development priorities and planned investments in the municipal plan?"
national_query = "What are Serbia's national priorities for sustainable development and public investment?"

municipal_response = run_query(
    municipal_query,
    mode="semantic",
    source_types={"municipal_development_plan"},
    municipality_id=SELECTED_MUNICIPALITY_ID,
)
print()
national_response = run_query(
    national_query,
    mode="semantic",
    source_types={"policy_document"},
)

Selected municipality: srb-beocin
mode=semantic total_results=1
query: What are the key local development priorities and planned investments in the municipal plan?
municipality_id: srb-beocin
category: None
  1. score=0.2719 title=serbia-serbia_municipal_development_plans-municipal_development_plan-beo-in-local-development-plan-97e370c718
     source_id=serbia-serbia_municipal_development_plans-municipal_development_plan-beo-in-local-development-plan-97e370c718 | chunk_id=serbia-serbia_municipal_development_plans-municipal_development_plan-beo-in-local-development-plan-97e370c718:0
     snippet=Dataset Family: serbia_municipal_development_plans Title: Beočin Local Development Plan Country: Serbia Municipality: Beočin District: South Bačka Source URL: http://beocin.rs/images/dokumenti/razno/PR_Grada_Beocin_2023-

mode=semantic total_results=5
query: What are Serbia's national priorities for sustainable development and public investment?
municipality_id: None
category: None
  1. score=0.

Tip: change `query`, `mode`, `source_types`, and `SELECTED_MUNICIPALITY_ID` in the last cell to run your own smoke experiments.

Useful source type filters:
- `{"municipal_development_plan"}`
- `{"policy_document"}`
- `{"project_page"}`
- `{"dataset"}`